# PyTorch Image Classification: 10-Class Image Recognition

This notebook builds an end-to-end image classifier for the supplied image archive and `Train.csv` / `Test.csv`. It is designed to run in Google Colab. The workflow mounts/downloads the data, checks the CSVs and image files, constructs PyTorch `Dataset`/`DataLoader` objects, applies training-time augmentation, selects hyperparameters with Optuna's Tree-structured Parzen Estimator (TPE), fine-tunes an ImageNet-pretrained ResNet-18, evaluates on a stratified holdout set, and writes a CSV containing predictions for the test IDs.

**Reproducibility and validation.** A fixed random seed is used. A stratified train/validation split helps keep the ten classes represented in both partitions. The validation set is used for model selection and checkpoint selection only; the held-out test labels are never required or used. After tuning, the selected settings are trained on all labeled images for the best epoch count observed during tuning.

**Expected deliverable:** `submission.csv`, with the same ID column/order as `Test.csv` and a predicted `label` column. This file is generated when the notebook is run with access to the course dataset.


## 1. Runtime and data access

Enable a GPU in **Runtime → Change runtime type → T4 GPU** if available. The dataset folder is shared through [Google Drive](https://drive.google.com/drive/folders/12OeYCMZuqaxQFqzppItp9qlcGsH3dRZ0?usp=sharing). The next cell tries to download that shared folder. If it is private or Google Drive blocks automated downloads, add the shared folder as a shortcut in your Drive, mount Drive below, then set `DRIVE_DATA_DIR` to the folder containing the archive/CSVs.


In [ ]:
%pip -q install optuna gdown


In [ ]:
from pathlib import Path
import os, random, zipfile, shutil, warnings
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import optuna

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
ImageFile.LOAD_TRUNCATED_IMAGES = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Dataset shared-folder ID from the assignment.
DRIVE_FOLDER_ID = '12OeYCMZuqaxQFqzppItp9qlcGsH3dRZ0'
DATA_DIR = Path('/content/image_classification_data')
DRIVE_DATA_DIR = None  # Example: '/content/drive/MyDrive/course_dataset'
DATA_DIR.mkdir(parents=True, exist_ok=True)
try:
    import gdown
    gdown.download_folder(id=DRIVE_FOLDER_ID, output=str(DATA_DIR), quiet=False, use_cookies=False)
except Exception as exc:
    warnings.warn(f'Automatic download did not complete: {exc}')

# Optional fallback for a private/shared-with-me folder: mount Drive and set DRIVE_DATA_DIR.
if DRIVE_DATA_DIR:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path(DRIVE_DATA_DIR)
    if not DATA_DIR.exists(): raise FileNotFoundError(f'DRIVE_DATA_DIR does not exist: {DATA_DIR}')
print('Searching for files under:', DATA_DIR)


### Find and extract the archive

The code searches recursively for the two CSV files and extracts any zip archives into the working directory. If multiple files named `Train.csv` or `Test.csv` exist, the first match is used; inspect the printed paths and adjust `DATA_DIR` if needed.


In [ ]:
# Extract archives (avoid repeatedly re-extracting an archive in this runtime).
zip_paths = list(DATA_DIR.rglob('*.zip'))
for archive in zip_paths:
    destination = DATA_DIR / (archive.stem + '_extracted')
    if not destination.exists():
        print('Extracting:', archive)
        with zipfile.ZipFile(archive) as zf: zf.extractall(destination)

train_csvs = list(DATA_DIR.rglob('Train.csv'))
test_csvs = list(DATA_DIR.rglob('Test.csv'))
if not train_csvs or not test_csvs:
    raise FileNotFoundError('Could not locate Train.csv and Test.csv. Check the download or set DRIVE_DATA_DIR to the correct Drive folder.')
TRAIN_CSV, TEST_CSV = train_csvs[0], test_csvs[0]
print('Train CSV:', TRAIN_CSV, '| Test CSV:', TEST_CSV)
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
print('Train shape:', train_df.shape, '| Test shape:', test_df.shape)
display(train_df.head()); display(test_df.head())


## 2. Inspect metadata and map IDs to images

The training table must have an image-ID column and a `label` column; the test table must have the corresponding ID column. The ID column is inferred as the first common column other than `label`, and can be overridden explicitly below. Image files are indexed by filename stem, so IDs such as `123` match `123.jpg` (or another common image extension) even when the image files are nested in folders.


In [ ]:
LABEL_COL = 'label'
if LABEL_COL not in train_df.columns:
    raise ValueError(f'Train.csv must contain a {LABEL_COL!r} column; found {list(train_df.columns)}')
common_id_cols = [c for c in train_df.columns if c in test_df.columns and c != LABEL_COL]
if not common_id_cols: raise ValueError('Could not infer a shared image ID column; set ID_COL manually.')
ID_COL = common_id_cols[0]  # Change this if the CSV has more than one shared non-label column.
if train_df[ID_COL].isna().any() or test_df[ID_COL].isna().any(): raise ValueError('Image IDs cannot be empty.')
if train_df[LABEL_COL].isna().any(): raise ValueError('Training labels cannot be empty.')
class_values = sorted(train_df[LABEL_COL].unique().tolist(), key=lambda x: str(x))
if len(class_values) != 10: warnings.warn(f'Expected 10 classes, found {len(class_values)}: {class_values}')
label_to_index = {value: i for i, value in enumerate(class_values)}
index_to_label = {i: value for value, i in label_to_index.items()}
train_df['_target'] = train_df[LABEL_COL].map(label_to_index).astype(int)
print('ID column:', ID_COL, '| Classes:', class_values)
print(train_df[LABEL_COL].value_counts().sort_index())

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
image_files = [p for p in DATA_DIR.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
image_by_stem = {}
for p in image_files:
    key = p.stem.casefold()
    if key in image_by_stem:
        raise ValueError(f'Duplicate image filename stem {p.stem!r}: {image_by_stem[key]} and {p}. Resolve duplicate names to avoid incorrect ID matching.')
    image_by_stem[key] = p

def image_path_for(image_id):
    key = Path(str(image_id)).stem.casefold()
    if key not in image_by_stem: raise FileNotFoundError(f'No image found for ID {image_id!r}; checked {len(image_by_stem)} image files under {DATA_DIR}')
    return image_by_stem[key]

# Fail early if any CSV rows do not resolve to images.
train_paths = [image_path_for(v) for v in train_df[ID_COL]]
test_paths = [image_path_for(v) for v in test_df[ID_COL]]
print(f'Indexed {len(image_files)} images; resolved {len(train_paths)} train and {len(test_paths)} test rows.')
with Image.open(train_paths[0]) as im: print('Example image:', train_paths[0].name, im.size, im.mode)


## 3. Dataset, data augmentation, and validation split

`ImageTableDataset` subclasses `torch.utils.data.Dataset`; it opens images safely, converts them to RGB, and returns a transformed tensor plus the encoded target (or just the tensor for test data). Training augmentation uses random resized crops, horizontal flips, small rotations, and mild color jitter to reduce overfitting while retaining class identity. Validation/test transforms are deterministic. ImageNet normalization is used because the backbone starts from ImageNet-pretrained weights.


In [ ]:
IMAGE_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=12),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.12, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
] )
eval_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

class ImageTableDataset(Dataset):
    def __init__(self, frame, paths, transform=None, labeled=True):
        self.frame = frame.reset_index(drop=True)
        self.paths = list(paths)
        self.transform = transform
        self.labeled = labeled
    def __len__(self): return len(self.paths)
    def __getitem__(self, index):
        try:
            with Image.open(self.paths[index]) as image: image = image.convert('RGB')
        except Exception as exc:
            raise RuntimeError(f'Unable to read image {self.paths[index]}') from exc
        if self.transform: image = self.transform(image)
        if self.labeled: return image, int(self.frame.loc[index, '_target'])
        return image

train_indices, val_indices = train_test_split(
    np.arange(len(train_df)), test_size=0.20, random_state=SEED, stratify=train_df['_target']
)
print(f'Train/validation sizes: {len(train_indices)} / {len(val_indices)}')


## 4. Model and hyperparameter search (Optuna TPE)

The classifier is a ResNet-18 initialized with ImageNet weights, with its final fully connected layer replaced for this dataset. Transfer learning is a strong baseline when the labeled dataset is modest: early convolutional layers already encode useful visual patterns, while fine-tuning adapts the network to these classes.

For hyperparameter selection, Optuna's **TPE sampler** performs sequential model-based (Bayesian) optimization: after initial trials, it models promising and less-promising parameter regions and proposes learning rate, weight decay, dropout, and batch size values likely to improve validation accuracy. Each trial trains only on the training partition and is scored on the same stratified validation partition. Increase `N_TRIALS` / `TUNE_EPOCHS` if runtime permits; reduce them for a quicker smoke test.


In [ ]:
NUM_CLASSES = len(class_values)
NUM_WORKERS = 2 if DEVICE.type == 'cuda' else 0

def make_model(dropout=0.25, pretrained=True):
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(model.fc.in_features, NUM_CLASSES))
    return model

def make_loaders(batch_size, train_ids, valid_ids):
    train_ds = ImageTableDataset(train_df.iloc[train_ids], [train_paths[i] for i in train_ids], train_transform, True)
    valid_ds = ImageTableDataset(train_df.iloc[valid_ids], [train_paths[i] for i in valid_ids], eval_transform, True)
    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'), generator=generator)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
    return train_loader, valid_loader

def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train(); running_loss = 0.0; seen = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type=='cuda')):
            logits = model(images); loss = criterion(logits, labels)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        running_loss += loss.item() * labels.size(0); seen += labels.size(0)
    return running_loss / max(seen, 1)

@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval(); total_loss = 0.0; all_y, all_pred = [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type=='cuda')):
            logits = model(images)
            if criterion is not None: total_loss += criterion(logits, labels).item() * labels.size(0)
        all_y.extend(labels.cpu().numpy().tolist()); all_pred.extend(logits.argmax(1).cpu().numpy().tolist())
    return (accuracy_score(all_y, all_pred), total_loss / max(len(all_y), 1), all_y, all_pred)


In [ ]:
N_TRIALS = 8
TUNE_EPOCHS = 8
PATIENCE = 3

def objective(trial):
    lr = trial.suggest_float('lr', 1e-5, 2e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    dropout = trial.suggest_float('dropout', 0.0, 0.5)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    train_loader, valid_loader = make_loaders(batch_size, train_indices, val_indices)
    model = make_model(dropout=dropout, pretrained=True).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TUNE_EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type=='cuda'))
    best_acc, best_epoch, stale = 0.0, 1, 0
    for epoch in range(1, TUNE_EPOCHS + 1):
        train_epoch(model, train_loader, optimizer, criterion, scaler)
        val_acc, _, _, _ = evaluate(model, valid_loader, criterion)
        scheduler.step()
        trial.report(val_acc, epoch)
        if val_acc > best_acc:
            best_acc, best_epoch, stale = val_acc, epoch, 0
        else: stale += 1
        if trial.should_prune(): raise optuna.TrialPruned()
        if stale >= PATIENCE: break
    trial.set_user_attr('best_epoch', best_epoch)
    del model, train_loader, valid_loader
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return best_acc

sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=2))
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True, show_progress_bar=True)
print('Best validation accuracy:', study.best_value)
print('Best hyperparameters:', study.best_params)
print('Best trial epoch:', study.best_trial.user_attrs.get('best_epoch'))
display(study.trials_dataframe().sort_values('value', ascending=False).head())


## 5. Final validation evaluation

Refit once on the training portion using the best TPE-selected parameters, then report accuracy and per-class precision/recall on the untouched validation fold. The validation score is a useful estimate, not a guarantee of test-set accuracy.


In [ ]:
best = study.best_params
best_epochs = int(study.best_trial.user_attrs.get('best_epoch', TUNE_EPOCHS))
tune_train_loader, tune_val_loader = make_loaders(best['batch_size'], train_indices, val_indices)
validation_model = make_model(best['dropout'], pretrained=True).to(DEVICE)
optimizer = torch.optim.AdamW(validation_model.parameters(), lr=best['lr'], weight_decay=best['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(best_epochs, 1))
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type=='cuda'))
for epoch in range(best_epochs):
    loss = train_epoch(validation_model, tune_train_loader, optimizer, criterion, scaler)
    scheduler.step()
    print(f'Epoch {epoch+1}/{best_epochs} - train loss {loss:.4f}')
val_acc, val_loss, y_true, y_pred = evaluate(validation_model, tune_val_loader, criterion)
print(f'Holdout validation accuracy: {val_acc:.4f} | loss: {val_loss:.4f}')
print(classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)), target_names=[str(c) for c in class_values], zero_division=0))
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES))), cmap='Blues')
ax.set(title='Validation confusion matrix', xlabel='Predicted class', ylabel='True class')
ax.set_xticks(range(NUM_CLASSES), [str(c) for c in class_values], rotation=45, ha='right')
ax.set_yticks(range(NUM_CLASSES), [str(c) for c in class_values])
plt.colorbar(ax.images[0], ax=ax); plt.tight_layout(); plt.show()


## 6. Train on all labeled data and predict the test set

After selecting the model configuration and epoch count using validation, train a fresh model on all labeled rows. The validation data is included here because it is now part of the labeled training set; its labels were already used only for model selection. Test images are processed without augmentation.


In [ ]:
full_train_ds = ImageTableDataset(train_df, train_paths, train_transform, labeled=True)
full_train_loader = DataLoader(full_train_ds, batch_size=best['batch_size'], shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'), generator=torch.Generator().manual_seed(SEED))
test_ds = ImageTableDataset(test_df, test_paths, eval_transform, labeled=False)
test_loader = DataLoader(test_ds, batch_size=best['batch_size'], shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
final_model = make_model(best['dropout'], pretrained=True).to(DEVICE)
optimizer = torch.optim.AdamW(final_model.parameters(), lr=best['lr'], weight_decay=best['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(best_epochs, 1))
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type=='cuda'))
for epoch in range(best_epochs):
    loss = train_epoch(final_model, full_train_loader, optimizer, criterion, scaler)
    scheduler.step()
    print(f'Full-data epoch {epoch+1}/{best_epochs} - loss {loss:.4f}')

@torch.no_grad()
def predict(model, loader):
    model.eval(); predictions = []
    for images in loader:
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type=='cuda')):
            logits = model(images)
        predictions.extend(logits.argmax(1).cpu().numpy().tolist())
    return predictions

predicted_indices = predict(final_model, test_loader)
predicted_labels = [index_to_label[i] for i in predicted_indices]
# Preserve the input test table's ID values, column name, and row order; add the required label column.
submission = test_df.drop(columns=[LABEL_COL], errors='ignore').copy()
submission[LABEL_COL] = predicted_labels
OUTPUT_CSV = Path('/content/submission.csv')
submission.to_csv(OUTPUT_CSV, index=False)
print('Saved:', OUTPUT_CSV, '| shape:', submission.shape)
display(submission.head())


## 7. Download the submission file

The prediction file is created as `/content/submission.csv`. Run this cell to download it from Colab. Check that the output has the same number and order of rows as `Test.csv`, and that its header contains the ID column and `label`.


In [ ]:
from google.colab import files
files.download(str(OUTPUT_CSV))


## Method summary and rationale

1. **Data integrity:** CSV columns and missing values are checked, unique class labels are encoded consistently, and each train/test ID is matched to an image before training.
2. **Dataset API:** A custom `Dataset` reads RGB images from disk and returns image tensors and labels; `DataLoader` batches and shuffles training samples.
3. **Augmentation:** Random crop/scale, flips, rotations, and color perturbations provide realistic variation and regularization. Evaluation uses deterministic resizing/cropping.
4. **Transfer learning:** An ImageNet-pretrained ResNet-18 is fine-tuned with a replacement classification head, AdamW, cross-entropy with mild label smoothing, and a cosine learning-rate schedule.
5. **Hyperparameter selection:** Optuna TPE explores learning rate, weight decay, dropout, and batch size using stratified holdout accuracy. Pruning/early stopping limit unpromising or stale runs.
6. **Reporting and prediction:** Validation accuracy, class-wise metrics, and a confusion matrix are reported. The chosen configuration is retrained using all labeled data, and predictions are exported in test-row order.

For stronger results, allow more tuning trials/epochs, verify the validation confusion matrix for systematic errors, and make sure the shared folder contains the expected train/test images. Always retain a validation procedure when comparing changes; do not tune against the final test predictions.
